# 2 · Robustheitskurve: Wie viel Farbabweichung verträgt ein Modell?

**Idee:** Statt auf ein fremdes Spital zu warten, verändern wir die Färbung der Testbilder kontrolliert und messen, ab wann das Modell kippt. Weil nur die Farbe verändert wird, ist jeder Leistungsverlust eindeutig der Farbe zuzuordnen.

**Voraussetzung:** Notebook 1 wurde ausgeführt (Modelle in `models/`, Split und Stain-Parameter in `results/`).

**Zwei Verschiebungen**
1. **HED-Verschiebung mit Stärke σ:** Pro Slide werden H-, E- und DAB-Kanal zufällig skaliert und verschoben – wie ein anderes Labor. σ = 0 ist das Original.
2. **H × E-Raster:** H- und E-Intensität systematisch zwischen 60 % und 140 % – zeigt, auf welchen Farbstoff ein Modell empfindlich reagiert.

Getestet wird auf der ID-Validierung (Trainingsspitäler, andere Slides). Ohne künstliche Verschiebung gibt es dort keinen Domain Shift.

In [ ]:
# Einmalig installieren (oder: pip install -r requirements.txt)
# %pip install -r requirements.txt

import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import stainlib as sl
import trainlib as tl

DEVICE = tl.get_device()
print("Gerät:", DEVICE)

## Konfiguration

In [ ]:
DATA_ROOT = "data"
MODELS = ["baseline", "hed", "macenko"]   # aus Notebook 1, Seed 0
SIGMAS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]
N_EVAL = 3_000
WORKERS = 4
# Macenko-Filter für dunkle Pixel: None = ursprünglicher Lauf (Dateien ohne Zusatz),
# 60 = korrigierter Lauf. Braucht aus Notebook 1 den Lauf "dark60" (Modell + stain_params_dark60.pkl).
MACENKO_DARK = 60
SUF = "" if MACENKO_DARK is None else ("_dark" if MACENKO_DARK == 30 else f"_dark{MACENKO_DARK}")

wds, meta = tl.load_camelyon(DATA_ROOT, download=False)
split = tl.load_json("results/split.json")
sp = tl.load_pickle(f"results/stain_params{SUF}.pkl")
ref, slide_params = sp["ref"], sp["slide_params"]
tag = "-".join(map(str, split["train_hosp"]))

rng = np.random.default_rng(0)
eval_idx = rng.choice(split["id_val"], size=min(N_EVAL, len(split["id_val"])), replace=False)
eval_slides = np.unique(meta["slide"][eval_idx])

models = {}
for v in MODELS:
    m = tl.build_model("resnet18", pretrained=False)
    suf = SUF if "macenko" in v else ""      # baseline und hed hängen nicht vom Filter ab
    m.load_state_dict(tl.torch.load(f"models/{v}_seed0_train{tag}{suf}.pt", map_location="cpu"))
    models[v] = m
print("Macenko-Filter:", MACENKO_DARK, "| Modelle:", list(models), "| Testpatches:", len(eval_idx), "| Slides:", len(eval_slides))

## So sehen die Verschiebungen aus

In [ ]:
i0 = int(eval_idx[0]); s0 = int(meta["slide"][i0]); img0 = tl.load_image(wds, i0)
fig, ax = plt.subplots(1, len(SIGMAS), figsize=(2 * len(SIGMAS), 2.4))
for a, s in zip(ax, SIGMAS):
    a.imshow(sl.HEDShift(s, seed=1)(img0, i0, s0)); a.set_title(f"σ = {s}"); a.axis("off")
plt.suptitle("HED-Verschiebung (ein Seed)"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(len(FACTORS), len(FACTORS), figsize=(9, 9))
for r_, h in enumerate(FACTORS):
    for c_, e in enumerate(FACTORS):
        ax[r_, c_].imshow(sl.hed_scale(img0, h, e)); ax[r_, c_].axis("off")
        if r_ == 0: ax[r_, c_].set_title(f"E × {e}", fontsize=9)
        if c_ == 0: ax[r_, c_].text(-12, 48, f"H × {h}", rotation=90, va="center", fontsize=9)
plt.tight_layout(); plt.show()

## Sweep über σ

Für das Macenko-Modell werden die Slide-Parameter auf den *verschobenen* Bildern neu geschätzt – genau wie im Einsatz, wo ein neues Labor eigene Parameter bekommt. Das zeigt, ob die Normalisierung die Verschiebung zurückrechnet.

Neben Genauigkeit und AUC wird die **Flip-Rate** gemessen: der Anteil der Patches, deren Vorhersage sich gegenüber σ = 0 ändert.

In [ ]:
def evaluate_shift(perturb):
    out, preds = [], {}
    sp_shift = None
    for v, model in models.items():
        params = slide_params
        if "macenko" in v:
            if sp_shift is None:
                sp_shift = tl.fit_slide_params(wds, meta, slides=eval_slides, k=32, perturb=perturb, verbose=False,
                                                dark_thresh=MACENKO_DARK)
            params = sp_shift
        ds = tl.PatchDataset(wds, eval_idx, meta, mode=v, slide_params=params, ref=ref, perturb=perturb)
        p = tl.predict(model, ds, workers=WORKERS, device=DEVICE)
        preds[v] = p["prob"] >= 0.5
        out.append({"model": v, **tl.metrics(p)})
    return out, preds

rows, base_preds = [], None
for s in SIGMAS:
    r, preds = evaluate_shift(sl.HEDShift(s, seed=1) if s > 0 else None)
    if base_preds is None:
        base_preds = preds
    for row in r:
        row.update(sigma=s, flip_rate=float((preds[row["model"]] != base_preds[row["model"]]).mean()))
    rows += r
    print(f"σ = {s}: " + ", ".join(f"{x['model']} {x['acc']:.3f}" for x in r))
curve = pd.DataFrame(rows)
curve.to_csv(f"results/nb2_sigma_curve{SUF}.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for v, g in curve.groupby("model"):
    ax[0].plot(g["sigma"], g["acc"], marker="o", label=v)
    ax[1].plot(g["sigma"], g["auc"], marker="o", label=v)
    ax[2].plot(g["sigma"], g["flip_rate"], marker="o", label=v)
for a, t in zip(ax, ["Genauigkeit", "AUC", "Flip-Rate gegenüber σ = 0"]):
    a.set_xlabel("Stärke der Farbverschiebung σ"); a.set_title(t); a.grid(alpha=0.3)
ax[0].legend()
plt.tight_layout(); plt.savefig(f"results/nb2_robustness_curve{SUF}.png", dpi=200); plt.show()

## H × E-Raster

In [ ]:
grid_rows = []
for h in FACTORS:
    for e in FACTORS:
        r, _ = evaluate_shift(sl.HEDScaleShift(h, e) if (h, e) != (1.0, 1.0) else None)
        grid_rows += [dict(x, h=h, e=e) for x in r]
grid = pd.DataFrame(grid_rows)
grid.to_csv(f"results/nb2_he_grid{SUF}.csv", index=False)

fig, ax = plt.subplots(1, len(models), figsize=(5 * len(models), 4.2))
for a, v in zip(np.atleast_1d(ax), models):
    M_ = grid[grid["model"] == v].pivot(index="h", columns="e", values="acc").sort_index(ascending=False)
    im = a.imshow(M_.values, vmin=0.5, vmax=1.0, cmap="viridis")
    a.set_xticks(range(len(FACTORS)), [f"{x}" for x in M_.columns]); a.set_yticks(range(len(FACTORS)), [f"{x}" for x in M_.index])
    a.set_xlabel("Eosin-Faktor"); a.set_ylabel("Hämatoxylin-Faktor"); a.set_title(v)
    for (r_, c_), val in np.ndenumerate(M_.values):
        a.text(c_, r_, f"{val:.2f}", ha="center", va="center", fontsize=8, color="white" if val < 0.8 else "black")
fig.colorbar(im, ax=ax, shrink=0.8, label="Genauigkeit")
plt.savefig(f"results/nb2_he_grid{SUF}.png", dpi=200, bbox_inches="tight"); plt.show()

## Brücke zu Notebook 1: Welche künstliche Verschiebung entspricht einem echten fremden Spital?

Wir suchen das σ, bei dem das Baseline-Modell so viel verliert wie in Notebook 1 auf den fremden Spitälern. Das ergibt eine anschauliche Aussage wie „Ein fremdes Spital wirkt auf das Modell wie eine Farbverschiebung der Stärke σ ≈ …“. Achtung: Echte Spitäler unterscheiden sich nicht nur in der Farbe, deshalb ist das eine grobe Einordnung.

In [ ]:
nb1 = pd.read_csv("results/nb1_results.csv")
b = nb1[(nb1["variant"] == "baseline") & (nb1["seed"] == 0)]
ood_acc = b[b["eval"] != "id_val"]["acc"].mean()
g = curve[curve["model"] == "baseline"].sort_values("sigma")
if g["acc"].min() <= ood_acc:
    sig_eq = float(np.interp(ood_acc, g["acc"].values[::-1], g["sigma"].values[::-1]))
    print(f"Baseline auf fremden Spitälern: {ood_acc:.3f} → entspricht etwa σ ≈ {sig_eq:.2f}")
else:
    print(f"Baseline auf fremden Spitälern: {ood_acc:.3f}. Selbst σ = {g['sigma'].max()} senkt die Genauigkeit "
          f"nur auf {g['acc'].min():.3f} – der echte Shift ist mehr als nur Farbe.")